In [ ]:
from huggingface_hub import login, whoami
whoami()

!pip install wandb -Uq

import os, wandb

# ── dedicated workspace for final runs ────────────────────────────────
WANDB_PROJECT = "mdlm-sft-final"            # new project = fresh workspace
WANDB_GROUP   = "stage2-reasoning"           # optional: groups related runs in the UI

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

#wandb.login()

from datasets import Dataset, DatasetDict
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
import tempfile
from pathlib import Path
from datasets import DatasetDict, load_from_disk
from huggingface_hub import list_bucket_tree, download_bucket_files

BUCKET = "avgJo3/final-mdlm-sft-artifacts"

PREFIXES: dict[str, str] = {
    "base":      "20260714T050657-0e2e053501df5170/data_",
    "reasoning": "20260715T045644-7d0f95338c3101b3/data_",   #
}

CHAIN_STEMS = {
    "train":        "train",          # (γ=strict, σ=reload)
    "mix":          "mix",            # (γ=mix,    σ=reload)
    "ablation":     "ablation",       # (γ=strict, σ=continue)
    "mix-ablation": "mix-ablation",   # (γ=mix,    σ=continue)
}
ROUNDS = ["R0", "R1", "R2", "R3", "R4"]

# Retained tempdirs, keyed by checkpoint alias, so lazy Dataset reads stay valid.
_TEMPDIRS: dict[str, tempfile.TemporaryDirectory] = {}


def load_chains_for(ckpt_alias: str) -> dict[str, DatasetDict]:
    if ckpt_alias not in PREFIXES:
        raise KeyError(f"unknown checkpoint alias {ckpt_alias!r}; "
                       f"known aliases: {list(PREFIXES)}")
    prefix = PREFIXES[ckpt_alias]
    if prefix is None:
        raise ValueError(f"prefix for {ckpt_alias!r} is not yet configured")

    files = [
        item for item in list_bucket_tree(BUCKET, prefix=prefix, recursive=True)
        if item.type == "file"
    ]
    if not files:
        raise RuntimeError(f"no files found under prefix: {prefix}")

    tmp = tempfile.TemporaryDirectory(prefix=f"chains-{ckpt_alias}-")
    _TEMPDIRS[ckpt_alias] = tmp
    tmp_root = Path(tmp.name)

    downloads = [(f, str(tmp_root / f.path)) for f in files]
    download_bucket_files(BUCKET, files=downloads)

    data_root = tmp_root / prefix.rstrip("/").removesuffix("data_").rstrip("/")
    shared_r0_dir = data_root / "data_train-generated_r0"
    assert shared_r0_dir.exists(), f"shared R0 dir not found: {shared_r0_dir}"

    def load_chain(stem: str) -> DatasetDict:
        splits = {"R0": load_from_disk(str(shared_r0_dir))}
        for k in range(1, len(ROUNDS)):
            round_dir = data_root / f"data_{stem}-generated_r{k}"
            splits[f"R{k}"] = load_from_disk(str(round_dir))
        return DatasetDict(splits)

    return {cid: load_chain(stem) for cid, stem in CHAIN_STEMS.items()}


# --- usage: load one checkpoint's chains -------------------------------------
chains_by_ckpt: dict[str, dict[str, DatasetDict]] = {}
chains_by_ckpt["base"] = load_chains_for("base")
chains_by_ckpt["reasoning"] = load_chains_for("reasoning")

print(chains_by_ckpt["base"])
#print(chains_by_ckpt["reasoning"])

In [ ]:
def embed_fn(batch):
    return {
        "embeddings": model.encode(
            batch["completion"],
            show_progress_bar=False,
            normalize_embeddings=True,
        )
    }

for ckpt_alias, chains in chains_by_ckpt.items():
    for chain_id in list(chains.keys()):   # ← snapshot of keys
        print(f"[embed] {ckpt_alias}/{chain_id}")
        chains_by_ckpt[ckpt_alias][chain_id] = chains_by_ckpt[ckpt_alias][chain_id].map(
            embed_fn, batched=True
        )

del model

In [ ]:
"""
Fréchet distance between two Gaussian approximations of embedding clouds.

Numerical-stabilisation logic (offset, imaginary-part handling) is adapted
from `pytorch-fid` (Seitzer 2020, ISC License), the reference implementation
used in the image-generation literature.
"""

import numpy as np
from scipy.linalg import sqrtm


def fid_gaussian(
    E_P: np.ndarray,
    E_Q: np.ndarray,
    eps: float = 1e-6,
) -> dict:
    """
    FID = ||mu_P - mu_Q||^2 + tr(Sigma_P + Sigma_Q - 2 (Sigma_P Sigma_Q)^{1/2}).

    Returns both the total and the two additive components separately.
    """
    E_P = np.atleast_2d(np.asarray(E_P, dtype=np.float64))
    E_Q = np.atleast_2d(np.asarray(E_Q, dtype=np.float64))

    mu_P = E_P.mean(axis=0)
    mu_Q = E_Q.mean(axis=0)
    Sigma_P = np.cov(E_P, rowvar=False)
    Sigma_Q = np.cov(E_Q, rowvar=False)

    diff = mu_P - mu_Q

    # Matrix product square root — the numerically delicate step.
    # Following pytorch-fid: if sqrtm fails, offset covariances slightly and retry.
    covmean, _ = sqrtm(Sigma_P @ Sigma_Q, disp=False)

    if not np.isfinite(covmean).all():
        offset = np.eye(Sigma_P.shape[0]) * eps
        covmean, _ = sqrtm((Sigma_P + offset) @ (Sigma_Q + offset), disp=False)

    # Discard tiny imaginary components from numerical noise.
    if np.iscomplexobj(covmean):
        assert np.abs(covmean.imag).max() < 1e-2, "large imaginary component — check inputs"
        covmean = covmean.real

    mean_term = float(diff @ diff)
    cov_term  = float(np.trace(Sigma_P + Sigma_Q - 2 * covmean))

    return {
        "fid":       mean_term + cov_term,
        "mean_term": mean_term,
        "cov_term":  cov_term,
    }

In [ ]:
def fid_gaussian_fast(E_P, E_Q, eps=1e-6):
    E_P = np.atleast_2d(np.asarray(E_P, dtype=np.float64))
    E_Q = np.atleast_2d(np.asarray(E_Q, dtype=np.float64))
    mu_P, mu_Q = E_P.mean(0), E_Q.mean(0)
    P = E_P - mu_P;  Sigma_P = (P.T @ P) / (len(E_P) - 1)
    Q = E_Q - mu_Q;  Sigma_Q = (Q.T @ Q) / (len(E_Q) - 1)
    vals_P, vecs_P = np.linalg.eigh(Sigma_P)
    vals_P = np.maximum(vals_P, 0)
    S = vecs_P * np.sqrt(vals_P)
    M = S.T @ Sigma_Q @ S
    vals_M = np.maximum(np.linalg.eigh(M)[0], 0)
    covmean_trace = np.sum(np.sqrt(vals_M))
    mean_term = float((mu_P - mu_Q) @ (mu_P - mu_Q))
    cov_term  = float(np.trace(Sigma_P) + np.trace(Sigma_Q) - 2 * covmean_trace)
    return {"fid": mean_term + cov_term, "mean_term": mean_term, "cov_term": cov_term}

In [ ]:
import itertools
import numpy as np
import pandas as pd

# --- one primitive: bootstrap FID between two clouds --------------------
def boot_fid(E_A, E_B, B, rng):
    """Return B×3 array of (fid, mean_term, cov_term). A==B → null floor."""
    n = E_A.shape[0]
    out = np.empty((B, 3))
    same = E_A is E_B
    for b in range(B):
        ia = rng.integers(0, n, n)
        ib = rng.integers(0, n, n)
        r = fid_gaussian_fast(E_A[ia], (E_A if same else E_B)[ib])
        out[b] = r["fid"], r["mean_term"], r["cov_term"]
    return out


# --- summarize a B×3 array into a one-row dict --------------------------
def summarize(arr, label_dict):
    metrics = ["fid", "mean_term", "cov_term"]
    row = dict(label_dict)
    for i, m in enumerate(metrics):
        v = arr[:, i]
        row[f"{m}_mean"]  = v.mean()
        row[f"{m}_ci_lo"] = np.percentile(v, 2.5)
        row[f"{m}_ci_hi"] = np.percentile(v, 97.5)
    return row


# --- main loop ----------------------------------------------------------
def run(chains_by_ckpt, B=1000, seed=1729):
    rng = np.random.default_rng(seed)
    pair_rows, null_rows = [], []

    for ckpt, chains in chains_by_ckpt.items():
        # sanity: R0 identical within checkpoint
        r0s = [c["R0"]["embeddings"] for c in chains.values()]
        assert all(np.array_equal(r0s[0], r) for r in r0s[1:]), f"R0 mismatch in {ckpt}"

        rounds = sorted(next(iter(chains.values())), key=lambda s: int(s[1:]))

        # inter-chain FID (k >= 1)
        for A, B_ in itertools.combinations(chains, 2):
            for rk in rounds:
                if rk == "R0": continue
                E_A = np.asarray(chains[A][rk]["embeddings"], dtype=np.float64)
                E_B = np.asarray(chains[B_][rk]["embeddings"], dtype=np.float64)
                arr = boot_fid(E_A, E_B, B, rng)
                pair_rows.append(summarize(arr, dict(
                    ckpt=ckpt, pair=f"{A}__vs__{B_}", round=rk, k=int(rk[1:]))))

        # null floor (all k including k=0)
        for cid, chain in chains.items():
            for rk in rounds:
                E = np.asarray(chain[rk]["embeddings"], dtype=np.float64)
                arr = boot_fid(E, E, B, rng)
                null_rows.append(summarize(arr, dict(
                    ckpt=ckpt, chain=cid, round=rk, k=int(rk[1:]))))

    return pd.DataFrame(pair_rows), pd.DataFrame(null_rows)


# --- run ---------------------------------------------------------------
pair_summary, null_summary = run(chains_by_ckpt, B=50)
print(pair_summary.head())
print(null_summary.head())

In [ ]:
import itertools
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- one primitive: bootstrap FID between two clouds --------------------
def boot_fid(E_A, E_B, B, rng):
    """Return B×3 array of (fid, mean_term, cov_term). A==B → null floor."""
    n = E_A.shape[0]
    out = np.empty((B, 3))
    same = E_A is E_B
    for b in range(B):
        ia = rng.integers(0, n, n)
        ib = rng.integers(0, n, n)
        r = fid_gaussian_fast(E_A[ia], (E_A if same else E_B)[ib])
        out[b] = r["fid"], r["mean_term"], r["cov_term"]
    return out


# --- summarize a B×3 array into a one-row dict --------------------------
def summarize(arr, label_dict):
    metrics = ["fid", "mean_term", "cov_term"]
    row = dict(label_dict)
    for i, m in enumerate(metrics):
        v = arr[:, i]
        row[f"{m}_mean"]  = v.mean()
        row[f"{m}_ci_lo"] = np.percentile(v, 2.5)
        row[f"{m}_ci_hi"] = np.percentile(v, 97.5)
    return row


# --- main loop ----------------------------------------------------------
def run(chains_by_ckpt, B=1000, seed=1729):
    rng = np.random.default_rng(seed)
    pair_rows, null_rows = [], []

    # pre-count total jobs for the progress bar
    n_pairs = sum(
        len(list(itertools.combinations(chains, 2))) * (len(next(iter(chains.values()))) - 1)
        for chains in chains_by_ckpt.values()
    )
    n_nulls = sum(
        len(chains) * len(next(iter(chains.values())))
        for chains in chains_by_ckpt.values()
    )
    total = n_pairs + n_nulls

    with tqdm(total=total, desc="bootstrap FID", unit="job") as pbar:

        for ckpt, chains in chains_by_ckpt.items():
            # sanity: R0 identical within checkpoint
            r0s = [c["R0"]["embeddings"] for c in chains.values()]
            assert all(np.array_equal(r0s[0], r) for r in r0s[1:]), f"R0 mismatch in {ckpt}"

            rounds = sorted(next(iter(chains.values())), key=lambda s: int(s[1:]))

            # cache casts
            E = {
                cid: {rk: np.asarray(chain[rk]["embeddings"], dtype=np.float64)
                      for rk in rounds}
                for cid, chain in chains.items()
            }

            # inter-chain FID (k >= 1)
            for A, B_ in itertools.combinations(chains, 2):
                for rk in rounds:
                    if rk == "R0":
                        continue
                    arr = boot_fid(E[A][rk], E[B_][rk], B, rng)
                    pair_rows.append(summarize(arr, dict(
                        ckpt=ckpt, pair=f"{A}__vs__{B_}", round=rk, k=int(rk[1:]))))
                    pbar.set_postfix(ckpt=ckpt, pair=f"{A}vs{B_}", round=rk)
                    pbar.update(1)

            # null floor (all k including k=0)
            for cid in chains:
                for rk in rounds:
                    arr = boot_fid(E[cid][rk], E[cid][rk], B, rng)
                    null_rows.append(summarize(arr, dict(
                        ckpt=ckpt, chain=cid, round=rk, k=int(rk[1:]))))
                    pbar.set_postfix(ckpt=ckpt, chain=cid, round=rk)
                    pbar.update(1)

    return pd.DataFrame(pair_rows), pd.DataFrame(null_rows)


# --- run ---------------------------------------------------------------
pair_summary, null_summary = run(chains_by_ckpt, B=50)
print(pair_summary.head())
print(null_summary.head())

In [ ]:
print(null_summary.columns.tolist())
print(null_summary.head(10))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

PAIR_COLORS = [
    "#0072B2",
    "#B22222",
    "#009E73",
    "#7B2D8B",
    "#D55E00",
    "#117733",
]

NULL_COLOR = "#888888"
COL_CKPTS  = ["base", "reasoning"]

rounds = sorted(pair_summary["round"].unique(), key=lambda s: int(s[1:]))
x      = np.arange(len(rounds))

# ── null ceiling: pointwise max of fid_ci_hi across all chains ───────────────
null_ceiling = (
    null_summary
    .groupby(["ckpt", "round"])["fid_ci_hi"]
    .max()
    .reset_index()
    .rename(columns={"fid_ci_hi": "null_ceiling"})
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    sharey=True,
    constrained_layout=False,
)
fig.subplots_adjust(bottom=0.28, wspace=0.08)

for col_idx, ckpt in enumerate(COL_CKPTS):
    ax = axes[col_idx]

    # ── null ceiling line ─────────────────────────────────────────────────────
    nc = (
        null_ceiling[null_ceiling["ckpt"] == ckpt]
        .set_index("round")
        .reindex(rounds)["null_ceiling"]
        .values
    )
    ax.plot(
        x, nc,
        color=NULL_COLOR, linewidth=1.2, linestyle="--",
        zorder=2,
    )

    # ── inter-pair lines ──────────────────────────────────────────────────────
    ckpt_pairs = pair_summary[pair_summary["ckpt"] == ckpt]["pair"].unique()

    handles, labels = [], []

    for p_idx, pair in enumerate(ckpt_pairs):
        color = PAIR_COLORS[p_idx % len(PAIR_COLORS)]
        pdata = (
            pair_summary[
                (pair_summary["ckpt"]  == ckpt) &
                (pair_summary["pair"]  == pair) &
                (pair_summary["round"].isin(rounds))
            ]
            .set_index("round")
            .reindex(rounds)
        )

        y     = pdata["fid_mean"].values
        ci_lo = pdata["fid_ci_lo"].values
        ci_hi = pdata["fid_ci_hi"].values

        # whiskers
        ax.vlines(x, ci_lo, ci_hi,
                  color=color, linewidth=0.8, alpha=0.5, zorder=3)
        ax.plot(x, ci_lo, "_", color=color, markersize=4, alpha=0.5, zorder=3)
        ax.plot(x, ci_hi, "_", color=color, markersize=4, alpha=0.5, zorder=3)

        # mean line
        line, = ax.plot(
            x, y,
            color=color, linewidth=2.0, marker="o", markersize=5,
            markeredgecolor="white", markeredgewidth=0.7,
            zorder=4,
        )
        short = pair.replace("__vs__", " × ")
        handles.append(line)
        labels.append(short)

    # null ceiling handle
    null_line = mpl.lines.Line2D(
        [], [], color=NULL_COLOR, linewidth=1.2, linestyle="--"
    )
    handles.append(null_line)
    labels.append("null ceiling")

    ax.set_title(ckpt, fontweight="bold")
    ax.set_xlabel("Round")
    ax.set_xlim(0, len(rounds) - 1)
    ax.set_xticks(x)
    ax.set_xticklabels(rounds)
    if col_idx == 0:
        ax.set_ylabel("FID  (bootstrap mean + 95% CI)")

    ax.legend(
        handles, labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=3,
        framealpha=0.6,
        fontsize=7,
    )

fig.suptitle(
    "Inter-chain FID per model  ·  whiskers = 95% bootstrap CI  ·  dashed = null ceiling",
    fontsize=10,
)
plt.savefig("fid_drift.png", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# --- reuse the same primitives -----------------------------------------
def boot_fid(E_A, E_B, B, rng, fixed_ref=False):
    """
    If fixed_ref: only E_A is bootstrapped; E_B is used as-is every replicate.
    Otherwise: both are bootstrapped independently.
    """
    n = E_A.shape[0]
    out = np.empty((B, 3))
    same = E_A is E_B
    for b in range(B):
        ia = rng.integers(0, n, n)
        E_A_b = E_A[ia]
        if fixed_ref:
            E_B_b = E_B          # fixed reference
        else:
            ib = rng.integers(0, n, n)
            E_B_b = (E_A if same else E_B)[ib]
        r = fid_gaussian(E_A_b, E_B_b)
        out[b] = r["fid"], r["mean_term"], r["cov_term"]
    return out


def summarize(arr, label_dict):
    metrics = ["fid", "mean_term", "cov_term"]
    row = dict(label_dict)
    for i, m in enumerate(metrics):
        v = arr[:, i]
        row[f"{m}_mean"]  = v.mean()
        row[f"{m}_ci_lo"] = np.percentile(v, 2.5)
        row[f"{m}_ci_hi"] = np.percentile(v, 97.5)
    return row


def run_intra(chains_by_ckpt, B=1000, seed=1729):
    """
    Intra-chain drift FID with conditional bootstrap: D_0 is held fixed as
    the reference, only D_k (k>=1) is resampled per replicate. Matches the
    fixed-reference design in the methodology spec.

    Returns:
        drift_summary: one row per (ckpt, chain, round) for k >= 1
        null_summary : one row per (ckpt, chain, round) for all k
                       (two independent resamples of the same cloud)
    """
    rng = np.random.default_rng(seed)
    drift_rows, null_rows = [], []

    for ckpt, chains in chains_by_ckpt.items():
        rounds = sorted(next(iter(chains.values())), key=lambda s: int(s[1:]))

        for cid, chain in chains.items():
            E0 = np.asarray(chain["R0"]["embeddings"], dtype=np.float64)

            # intra-chain drift FID (k >= 1): conditional bootstrap, D_0 fixed
            for rk in rounds:
                if rk == "R0": continue
                Ek = np.asarray(chain[rk]["embeddings"], dtype=np.float64)
                arr = boot_fid(Ek, E0, B, rng, fixed_ref=True)   # <-- fixed D_0
                drift_rows.append(summarize(arr, dict(
                    ckpt=ckpt, chain=cid, round=rk, k=int(rk[1:]))))

            # null floor (all k including k=0): two independent resamples
            for rk in rounds:
                E = np.asarray(chain[rk]["embeddings"], dtype=np.float64)
                arr = boot_fid(E, E, B, rng, fixed_ref=False)    # <-- independent draws
                null_rows.append(summarize(arr, dict(
                    ckpt=ckpt, chain=cid, round=rk, k=int(rk[1:]))))

    return pd.DataFrame(drift_rows), pd.DataFrame(null_rows)


# --- run ---------------------------------------------------------------
drift_summary, null_summary_intra = run_intra(chains_by_ckpt, B=50)
print(drift_summary.head())
print(null_summary_intra.head())